# Algorithms 1 and 2: executable reference

## Purpose

This notebook is an executable companion to the manuscript's test-conditional mutant execution-planning procedure. It first reproduces the hand-checkable three-qubit GHZ walkthrough checks those gaps with separately constructed circuits, forces the signed pilot-estimation branch on a one-qubit example, and classifies actual finite-shot observations.

A practitioner should finish the notebook able to:

- distinguish an **execution-decision status** from the **evidence type** supporting it;
- understand why a mutant can be skipped for one fixed test without being program-equivalent;
- read ideal gaps, deterministic bounds, confidence intervals, error components, and within-evidence ranks; and
- recognize when an apparently zero pilot estimate is still too uncertain to justify a skip.

> **Scope.** The planner prioritizes candidate executions for one already-declared test. It does not generate a mutation population, establish test-suite adequacy, or replace the independent survivor census described in the empirical protocol.

## How the algorithms fit together

1. **Algorithm 1** applies the ordered planning rules: pinned transpiler equality, the periodic rotation threshold ceiling, structural backward-light-cone exclusion, effect evaluation, threshold classification, and ranking.
2. **Algorithm 2** obtains the strongest supported effect evidence for a surviving candidate: exact, exact observable-side only, state-dependent perturbative, operator proxy, pilot-estimated, or unavailable.

The separate unknown-source discrimination floor is disabled by default; opt in with `planner.enable_discrimination_budget=true` or `PlannerSettings(enable_discrimination_budget=True)`. The report records `discrimination_budget_enabled`.

The decision status answers **what the planner recommends doing next**. The evidence type and provenance answer **what calculation or certificate supports that recommendation**. These fields must remain separate.

## Running the notebook

Launch JupyterLab from the code project root (the directory containing `pyproject.toml`) with `jupyter lab notebook/algorithms_1_2_demo.ipynb`. Select the Python kernel from the environment where the package is installed, then use **Run > Run All Cells** to generate the example outputs.

## Walk-through 1: fixed-test GHZ planning

The planning problem uses the three-qubit circuit `Ry(pi/2, q0); CX(q0, q1); CX(q0, q2)`, input state `|000>`, and the single output observable `Z` on qubit 1. The declared detection threshold is `0.1`; the retained `ExecutionBudget` specifies 10,000 copies and error probability `0.05` for the optional discrimination policy, which is disabled in this walkthrough.

The eight candidates are intentionally small and hand-checkable:

| ID | Declared mutation | Expected first decisive rule |
|---|---|---|
| `m1` | Delete `CX(q0, q2)` | Outside the backward light cone of `Z1` |
| `m2` | Append `X` on qubit 2 | Outside the backward light cone of `Z1` |
| `m3` | Append `Z` on qubit 1 | Exact commutation certificate |
| `m4` | Append `X` on qubit 1 | Certified state-dependent cancellation |
| `m5` | Reverse `Ry(pi/2)` to `Ry(-pi/2)` | Certified cancellation at this strength |
| `m6` | Add a `0.01`-radian angle offset | Strength-only gap ceiling below the test threshold |
| `m7` | Add a `pi/2` angle offset | Exact predicted gap above threshold |
| `m8` | Delete `CX(q0, q1)` | Exact predicted gap above threshold |

The Hydra override `planner.enable_equivalence=false` is deliberate for this walkthrough: it exposes the manuscript's structural, algebraic, and threshold branches directly. For operational use, keep equivalence filtering enabled and pin its transpiler settings.

In [ ]:
from hydra import compose, initialize_config_module
from quantum_mutant_planner.cli import settings_from_config
from quantum_mutant_planner.examples import build_ghz_problem
from quantum_mutant_planner.planner import plan_mutant_executions

with initialize_config_module(config_module="quantum_mutant_planner.conf", version_base=None):
    config = compose(config_name="config", overrides=["planner.enable_equivalence=false"])
settings = settings_from_config(config)
ghz_records = plan_mutant_executions(build_ghz_problem(), settings)

### Reading the result rows

Each tuple below has the form `(candidate_id, decision_status, evidence_type, gap, deterministic_interval, rank)`.

- `gap` is the absolute original-versus-mutant expectation shift when the planner computed one. `None` means that an earlier decisive rule made a gap calculation unnecessary.
- `rank` is assigned only to candidates retained for execution and only within a comparable evidence type. `None` means that the candidate was not ranked.
- A status beginning with `skip` is conditional on this fixed test, threshold, and explicitly enabled policy unless its provenance establishes full semantic equivalence.
- `deterministic_interval` bounds the ideal gap without a sampling claim; `interval` is reserved for pilot uncertainty. Neither is an exact point gap.

In [ ]:
[(record.candidate_id, record.status.value, record.evidence.evidence_type.value, record.evidence.gap, record.evidence.deterministic_interval, record.rank) for record in ghz_records]

### Interpreting the GHZ output

The rows show the ordered planner reducing eight declared candidates to two executions:

- `m1` and `m2` receive structural zero certificates because their changed qubits cannot influence the measured observable.
- `m3` receives an algebraic zero certificate because its relative action commutes with the propagated observable.
- `m4` and `m5` receive `skip--certified zero` with state-dependent cancellation provenance. The example uses explicit exact angles and the symbolic path independently compares the complete circuits. Ordinary floating-point angles cannot supply these algebraic certificates.
- `m6` is skipped before circuit analysis because `2 * abs(sin(0.01 / 2)) < 0.1`. Its `deterministic_interval` is approximately `[0, 0.01]`, while `gap` remains `None`. This small offset is a sensitivity probe, not an empirically justified fault model.
- `m7` and `m8` have exact gaps of approximately `1`, exceed the threshold `0.1`, and are the only ranked execution candidates.

> **Practitioner takeaway.** None of the test-relative skips by itself proves that the mutant program is semantically equivalent to the original. In a suite-adequacy study, non-equivalent skipped mutants remain in the independently fixed population denominator; their eventual survivor or kill labels follow the prespecified detection rule, not this planning disposition.

### Independent check of the manuscript

The following calculation rebuilds the original and two mutant circuits directly with Qiskit gates. It does not call the planner's local response formulas or mutation factories. Agreement up to roundoff checks the numerical prediction; the preceding exact-input comparison establishes the zero identity. A discrepancy requires investigation, not automatic adjustment of either the manuscript or a test expectation.


In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector

original = QuantumCircuit(3)
original.ry(np.pi / 2, 0)
original.cx(0, 1)
original.cx(0, 2)
m4 = original.copy()
m4.x(1)
m5 = QuantumCircuit(3)
m5.ry(-np.pi / 2, 0)
m5.cx(0, 1)
m5.cx(0, 2)
observable = SparsePauliOp("IZI")
mean = Statevector.from_instruction(original).expectation_value(observable)
independent_gaps = {
    name: float(abs(mean - Statevector.from_instruction(mutant).expectation_value(observable)))
    for name, mutant in [("m4", m4), ("m5", m5)]
}
assert all(gap < 1e-12 for gap in independent_gaps.values())
assert all(record.status.value == "skip--certified zero" for record in ghz_records if record.candidate_id in independent_gaps)
independent_gaps


## Walk-through 2: signed pilot estimation

The second example prepares `|+>` with a Hadamard gate, inserts an `X`-axis rotation of `pi/2`, and measures `Z`. It deliberately sets `exact_state_max_qubits=0` and `symbolic_max_qubits=0`, forcing Algorithm 2 to use the pilot branch even though this one-qubit state is classically tractable.

The pilot uses randomized local-Pauli measurements with a fixed seed. Its deliberately loose target error of `1.0` keeps the demonstration small enough to run interactively; it is not a recommended production accuracy. In practice, choose the retained-term cap, pilot accuracy, and shot ceiling on disjoint calibration circuits and charge their costs to the planning budget.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Pauli, SparsePauliOp, Statevector
from quantum_mutant_planner.models import ExecutionBudget, PilotSettings, PlannerSettings, PlanningProblem, QuantumTest
from quantum_mutant_planner.mutations import pauli_rotation_mutation

circuit = QuantumCircuit(1)
circuit.h(0)
candidate = pauli_rotation_mutation(circuit, "pilot-demo", 1, Pauli("X"), 0.0, np.pi / 2, qubits=(0,), insert=True)
pilot_problem = PlanningProblem(circuit, QuantumTest(Statevector.from_label("0"), SparsePauliOp("Z"), 0.1, 2.0), (candidate,), ExecutionBudget(10000, 0.05))
pilot_settings = PlannerSettings(exact_state_max_qubits=0, symbolic_max_qubits=0, exact_simulation_max_qubits=1, enable_equivalence=False, pilot=PilotSettings(target_error=1.0, failure_probability=0.1, max_shots=1000, seed=7))
pilot_record = plan_mutant_executions(pilot_problem, pilot_settings)[0]
pilot_record.to_dict()

### Interpreting the current pilot output

The point estimate is `gap = 0.0`, but the simultaneous interval is `[0.0, 2.0]`. Because that interval contains the threshold `0.1`, the correct decision is `execute--threshold unresolved`, not a zero-effect skip.

The remaining fields explain why:

- `evidence_type = pilot-estimated` identifies a finite-sample estimate rather than an exact gap.
- `sampling_error = 2.0` dominates the interval, while `truncation_error` records the discarded Pauli mass, including numerical cutoffs; any roundoff-scale contribution is retained separately.
- `zero_provenance = none` correctly prevents a finite-sample zero estimate from becoming an exact certificate.
- The warning records the prefix shots used. They are shared across the target observables in this call; reuse across candidates remains a future extension. `failure_probability = 0.1` is the allocation for this one-candidate run.
- `rank = 1` is only a within-evidence rank; here there is a single pilot-estimated candidate, so it is not a meaningful cross-method comparison.

> **Practitioner takeaway.** Base threshold decisions on the complete interval and its separately reported sampling and truncation components. Never promote a zero point estimate, a truncated observable-side result, or an operator proxy into an exact zero claim.

## Walk-through 3: observed finite-shot classification

Planning predictions and observed detections are separate results. For an illustrative GHZ comparison, sample the original `Z1` outcomes independently with equal probabilities; the retained `m7` mutant has deterministic outcome `-1`. Prespecify 4,096 shots of each circuit and a test error allocation of `0.05`. This allocation is separate from the pilot budget and the discrimination error requirement.

`classify_fixed_shot` returns the observed absolute gap, the clipped confidence interval, and `below threshold`, `detected`, or `unresolved`. It requires independent observations within each comparison and a fixed sample size. Reusing original samples across prespecified comparisons is allowed when their error allocations sum to the family-wise budget; repeated inspection needs separate sequential control.


In [ ]:
from dataclasses import asdict
from quantum_mutant_planner import ObservationStatus, classify_fixed_shot

observation_rng = np.random.default_rng(20261011)
original_outcomes = observation_rng.choice([-1.0, 1.0], size=4096)
mutant_outcomes = np.full(4096, -1.0)
observed = classify_fixed_shot(
    original_outcomes, mutant_outcomes,
    outcome_bounds=(-1.0, 1.0), threshold=0.1, failure_probability=0.05,
)
assert observed.status is ObservationStatus.DETECTED
asdict(observed)


## Interpretation boundary

These calculations use ideal, finite-dimensional unitary simulation. They do not cover device noise, mid-circuit measurement, reset, classical control, channels, or the cost and variability of real hardware execution.

The examples demonstrate the reference implementation; they are not completed empirical evidence for the manuscript's follow-on execution-planning or pilot-estimation studies. Any operational evaluation must use a frozen candidate population and test suite, retain separate decision and evidence fields, report every declared threshold and error allowance, and compare predicted decisions with held-out realized effects.

Analytic deterministic bounds are evaluated in floating-point arithmetic and retain that provenance. The planner allocates the configured pilot failure budget across all declared candidates; a direct effect-evaluation call uses a per-call budget. No finite pilot certifies an exact zero.

## Adapting the example to your own circuit

1. Construct a `PlanningProblem` with the original Qiskit circuit, a fixed input state and Hermitian observable, a declared threshold, a candidate tuple, and an execution budget.
2. Use the mutation factories in `quantum_mutant_planner.mutations` so support, site, relative-unitary, and left-multiplication metadata remain consistent.
3. Start from `PlannerSettings()` and change computational caps only when you can justify the resulting evidence branch and error accounting.
4. Call `plan_mutant_executions(problem, settings)` and persist the full `PlanningRecord` objects rather than only the final ranks.
5. Inspect status, evidence type, provenance, interval, error components, and warnings before deciding which candidates to execute.